## Music Recommendation Algorithim - Final Report

This report answers the 5 required questions based on the findings from the EDA, cleaning, and modeling notebooks. 

## 1. Which insights did you gain from your EDA? Were any columns highly correlated? If so, name them.

From the EDA I learned a few key things about this dataset:

- The dataset has 28,362 songs spanning from 1950 to 2019 with zero null values and zero duplicates so it was already pretty clean to start with. 
- The genre distribution is imbalances with pop having the most songs (7,038) and hip hop having the fewest (904).
- The most common topics are sadness (6,094), violence (5,707), and world/life (5,419). The least common topic is feelings(612).
- All of the topic score features are heavily right-skewed meaning most songs score very low on any given topic with only a few scoring high. This makes sense because a song is usually only about one or two themes at most.
- Hip hop stood out as having by far the highest average obscene score(0.411) compared to other genres. Rock had the highest violence score(0.181)
- Lyrics have gotten longer over time going from an average of about 50 words in the 1950s to over 90 words by 2015. 
- Reggae and hip hop has the longest average lyrics (~98 words) while country had the shortest (~63 words)

When I checked for highly correlated columns using a threshold of |r| > 0.5, no feature paris exceed that threshold. The strongest corelation I found was between obscene and len at 0.44. However when I checked release_date against age I found a perfect negative correlation of -1.0 which makes sense since age is directly calculated from release_date.

All Kruskal-Wallis tests showed that every single topic score differs significantly across genres (p < 0.001 for all) which confirmed that these features carry meaningful information about the differences between genres. 

## 2. How did you determine which columns to drop or keep? If your EDA informed this process, explain which insights you used to determine which columns were not needed.

I dropped the following columns and here is my reasoning for each:

- **Unnamed: 0** - This was just a leftover index column from the original csv file. It has no predictive value.
- **artist_name** - This is a categorical identifier. KMeans needs numeric data and the artist name does not help with clustering songs by their lyrical features.
- **track_name** - Same reason as artist_name. It is just a label not a feature.
- **lyrics** - The project instructions said to drop this column for now. The raw text data would need separate processing like Word2Vec which is handled in a separate bonus notebook.
- **genre** - This is a categorical variable. While it could be encoded I decided not to include it since the goal is to discover clusters from the lyrical features themselves not to cluster by genre.
- **topic** - This column is a categorical label that appears to be derived from whichever topic score is highest for that song. Including it alongside the numeric topic scores would be redundant.
- **release_date** - From the EDA I discovered that release_date and age have a perfect negative correlation of -1.0. This means they contain the exact same information. I kept age because it is already normalized between 0 and 1 which is better for clustering.

After dropping these columns I was left with 17 numeric features: len, 15 topic scores (dating, violence, world/life, night/time, shake the audience, family/gospel, romantic, communication, obscene, music, movement/places, light/visual perceptions, family/spiritual, sadness, feelings), and age.

## 3. What was the optimal number of clusters in your cluster model? Explain how you determined this value.

The optimal number of clusters was **k=10** based on the silhouette score analysis.

I used two methods to determine this:

**Elbow Method:** I plotted the inertia (within-cluster sum of squares) for k values from 2 to 10. The curve decreased steadily without a clear sharp elbow point which is common with real world data where clusters are not perfectly separated. The curve did start to flatten somewhat around k=5-6 but there was no obvious cutoff.

**Silhouette Score:** I calculated the silhouette score for each value of k from 2 to 10. The silhouette score measures how similar each data point is to its own cluster compared to other clusters with higher scores being better. The scores increased from 0.0860 at k=2 to 0.1813 at k=10. There was a noticeable jump at k=8 (0.1766) and the highest score was at k=10 (0.1813).

I went with k=10 since it had the highest silhouette score. The overall silhouette scores are relatively low (0.18) which is normal for this type of high-dimensional data where songs naturally share characteristics across different themes. The clusters still capture meaningful patterns as shown in the cluster analysis.

## 4. Take a look at the respective songs that fell into your clusters. Describe these clusters in human terms.

Based on the average feature values and sample songs from each cluster here is how I would describe them:

- **Cluster 0** (1,110 songs) - World/life and violence themed songs with moderate lyrics length (~67 words). These are songs about life experiences with some edge to them. Mix of reggae, pop, and jazz. Example: Ziggy Marley - Changes.

- **Cluster 1** (4,991 songs) - High violence songs (0.437 avg). This is the largest violence-heavy cluster with songs across all genres especially rock and pop. These songs have aggressive or intense lyrical content. Example: Bring Me the Horizon - The House of Wolves.

- **Cluster 2** (2,319 songs) - High obscene songs (0.413 avg) with older age (0.537). These tend to be older songs with explicit or provocative lyrical content. Example: Sam Cooke - Having a Party.

- **Cluster 3** (5,396 songs) - Sadness-heavy songs (0.439 avg sadness). The largest cluster with a strong sadness theme. Country and pop dominate here. Songs about heartbreak and loss. Example: Shania Twain - When He Leaves You.

- **Cluster 4** (1,695 songs) - Romantic and older songs. Highest romantic score (0.405) and high age score (0.592) meaning these tend to be older love songs. Example: Paul Anka - Eso Beso.

- **Cluster 5** (808 songs) - The smallest cluster. Sadness-focused with moderate scores across the board. These songs have a more balanced emotional profile. High age (0.462) suggesting older melancholy songs.

- **Cluster 6** (1,922 songs) - World/life songs with high age (0.452). These are songs about broader life themes and the world around us. Mixed genres. Example: Randy Travis - Messin' With My Mind.

- **Cluster 7** (4,459 songs) - Long explicit songs. By far the highest lyrics length (116 words avg) and highest obscene score (0.464). This cluster is dominated by hip hop (673 songs in this cluster alone which is most of all hip hop). These are wordy songs with explicit content. Example: Rage Against the Machine - Pistol Grip Pump.

- **Cluster 8** (4,744 songs) - World/life and sadness blend (0.433 world/life). These songs deal with life themes and have a slightly sad undertone. Lots of country and pop. Newer songs (age 0.449).

- **Cluster 9** (918 songs) - High night/time (0.104) and world/life (0.104) with the highest dating score across all clusters (0.114). These songs tend to deal with nightlife, dating, and social situations. Example: Godsmack - Immune.

## 5. Take a look at the clusters that your algorithm assigned to your test samples. Based on these clusters, which songs would you recommend to this user?

The test dataset contains 10 songs from a hypothetical user. Here is how they were assigned:

| Song | Artist | Genre | Cluster |
|------|--------|-------|---------|
| Immune | Godsmack | Rock | 9 |
| Second Chance | Dennis Brown | Reggae | 6 |
| Sister Luck | The Black Crowes | Pop | 1 |
| Your Cheating Heart | Jerry Lee Lewis | Pop | 3 |
| Eso Beso | Paul Anka | Pop | 4 |
| Silencio | Noro Morales | Jazz | 1 |
| Pistol Grip Pump | Rage Against the Machine | Rock | 7 |
| Railway and Gun | Taste | Blues | 3 |
| Messin' With My Mind | Randy Travis | Country | 6 |
| Playing God | Paramore | Pop | 1 |

The user's songs are spread across 6 different clusters (1, 3, 4, 6, 7, 9) with **Cluster 1 being the most common** appearing 3 times. Cluster 1 is the violence-themed cluster with songs that have aggressive or intense lyrical content.

Based on this I would recommend songs from **Cluster 1** since that is where most of the user's listening falls. Here are 10 recommendations:

1. Wilco - Via Chicago (pop)
2. Bring Me the Horizon - The House of Wolves (rock)
3. Anthony B - Cold Feet (reggae)
4. Josh Turner - Gravity (country)
5. Frankie Lymon & the Teenagers - Who Put the Bomp (blues)
6. The Tymes - Goodnight My Love (blues)
7. Wire - Silk Skin Paws (blues)
8. Milky Chance - Cocoon (rock)
9. Cody Johnson - Billy's Brother (country)
10. Passafire - Growing Up (reggae)

These songs share the same lyrical profile as the user's most frequently clustered songs. The recommendations span multiple genres which makes sense since the clustering is based on lyrical features not genre. This means the user might discover new music across different genres that still matches the lyrical themes they tend to enjoy.